        # 🏕️ L00　新手村：Colab 操作與通關方式
        **Python 冒險之旅 2026**　｜　Day 1（08/29 六）🏝️ 起始之島　｜　新手村　｜　🏅 50 XP

        📖 對應教科書：第 1 章 1.6–1.7（本課程以 Colab 取代 Spyder）


        ### 🎯 這一關你會學到
        - 會開啟 Colab、執行程式格、儲存副本到雲端硬碟
- 認識程式格、文字格與執行順序
- 學會取得通關密語

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L00"
_SALT = "python-quest-2026-datama"
_TASKS = ["0-1", "0-2", "0-3"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_0_1(run):
    out, ns = run()
    return 出現(out, "Hello")
任務定義("0-1", _check_0_1, 提示="直接按程式格左邊的 ▶ 就可以了。")

def _check_0_2(run):
    out, ns = run()
    if "___" in out:
        return (False, "還沒把 ___ 換成你的暱稱喔。")
    return 出現(out, "我是勇者")
任務定義("0-2", _check_0_2, 提示="只要改引號裡面的文字，引號本身要保留。")

def _check_0_3(run):
    out, ns = run()
    ok = 出現(out, "冒險的第一天") and 出現(out, "Python好好玩") and len(行列表(out)) >= 2
    return ok
任務定義("0-3", _check_0_3, 提示="每一行 print 會印出一行；括號與引號要成對。")


## 🏕️ 歡迎來到新手村！

你現在看到的這個網頁叫做 **Google Colab**（Colaboratory）。它是 Google 提供的免費線上 Python 環境，
**不用安裝任何軟體**，只要有瀏覽器和 Google 帳號就能寫程式。這 6 天我們都會在這裡冒險。

### Colab 三件事
| 你會看到 | 它是什麼 | 怎麼用 |
|---|---|---|
| 灰色方框，左邊有 ▶ | **程式格（Code cell）** | 點 ▶ 或按 `Shift + Enter` 執行 |
| 白底文字（像這一段） | **文字格（Text cell）** | 說明用，讀就好；連點兩下可以編輯 |
| 左上角「＋ 程式碼」「＋ 文字」 | 新增格子 | 想多寫幾行程式就按它 |

> 🔑 **超重要**：Colab 會記住「你執行過的格子」。如果上面的格子沒執行，下面用到它的東西就會出錯。
> 迷路時用選單「執行階段 → 全部執行」最保險。

## 第 1 步：執行你的第一格程式
把滑鼠移到下面的程式格，點左邊的 ▶（第一次會等幾秒鐘「連線」）。

In [ ]:
print("哈囉，Colab！我準備好要冒險了 🚀")
print(1 + 1)

看到下面出現文字和 `2` 了嗎？恭喜，你已經讓電腦聽你的話了。

## 第 2 步：把這本筆記本存成自己的
點選上方選單 **「檔案」→「在雲端硬碟中儲存副本」**。
之後你做的修改都會存在自己的 Google 雲端硬碟裡（資料夾叫 `Colab Notebooks`）。

## 第 3 步：認識「任務」與「檢查」
每一關會有幾個 **🎯 任務**。任務的程式格第一行有一個標記（例如 `# 🎯 任務 0-1`），**請不要刪掉它**，
檢查工具靠這一行找到你的程式。做法永遠是：

1. 在任務格寫程式 → 按 ▶ 執行
2. 執行任務格**下面那一格** `檢查("0-1")` → 看到 ✅ 或 ❌
3. ❌ 的話依提示修改，重做 1、2

### 🎯 任務 0-1　執行就對了

這一格已經幫你寫好了，你只要 **執行它**，然後執行下面的檢查格。

In [ ]:
# 🎯 任務 0-1　執行就對了（請保留這一行）
print("Hello, Python 冒險之旅！")

In [ ]:
檢查("0-1")   # ◀ 執行這一格，看看任務 0-1 有沒有過關

### 🎯 任務 0-2　改一改：報上名來

把程式裡的 `___` 換成你的暱稱（要和你在入口網頁登錄的一樣），讓畫面印出 `我是勇者 xxx`。

In [ ]:
# 🎯 任務 0-2　改一改：報上名來（請保留這一行）
print("我是勇者 ___")

In [ ]:
檢查("0-2")   # ◀ 執行這一格，看看任務 0-2 有沒有過關

### 🎯 任務 0-3　多印一行

在下面的程式格**再加一行**程式，印出 `Python 好好玩`（提示：複製上一行再改文字）。

In [ ]:
# 🎯 任務 0-3　多印一行（請保留這一行）
print("今天是冒險的第一天")
# 在這一行下面加上一行 print(...)

In [ ]:
檢查("0-3")   # ◀ 執行這一格，看看任務 0-3 有沒有過關

## 第 4 步：常見的三個小狀況
| 狀況 | 原因 | 解法 |
|---|---|---|
| 左邊一直轉圈圈 | 正在連線或執行 | 等幾秒；太久就選「執行階段 → 中斷執行」 |
| 出現紅色錯誤訊息 | 程式有打錯字或少了符號 | 看最後一行訊息，通常會告訴你第幾行有問題 |
| 說某個名稱 `is not defined` | 上面的格子沒先執行 | 先執行上面的格子（或「全部執行」） |

---
## 🔑 通關密語
　這是最簡單的一關，先熟悉流程就好！
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🐍 L01 Python 初體驗** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L01_python_first_steps.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/